In [13]:
import os
import certifi 
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

/home/rare/miniconda3/envs/langagent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.agents import create_react_agent, AgentExecutor

In [ ]:
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [7]:
search_tool = TavilySearchResults(max_results=2)

In [8]:
result=search_tool.invoke('Give  me the latest news on AI')
result 

[{'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': "Now, let's explore the latest news for AI advancements and developments from May, June and July, 2026.\n\n### (AI Breakthrough) Innovative Eyewear Brings Claude AI to All Lucyd Smart Glasses\n\nDate: July 10, 2026\n\nSummary: Innovative Eyewear announced a Claude AI integration across its entire Lucyd smart eyewear lineup, available free to all customers via the Lucyd app. Users can choose between Claude and ChatGPT and switch models mid-conversation without losing context. Features include image and document analysis, AI image generation, incognito mode, and cited web sources. A hands-free version enabling Claude queries with the phone locked is planned for late Q3 2026, supported by the company's pending patent on multi-AI access glasses.\n\nSource: PR Newswire ↗ [...] Date: May 21, 2026\n\nSummary: SoundHound AI announced the acquisition of conversational AI platform LivePerson, combining its voice an

In [14]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

ValidationError: 1 validation error for ChatGoogleGenerativeAI
__root__
  Did not find google_api_key, please add an environment variable `GOOGLE_API_KEY` which contains it, or pass `google_api_key` as a named parameter. (type=value_error)

In [10]:
response = llm.invoke("Tell me a joke about AI")
response

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## ReAct research agent

`handle_parsing_errors` sends malformed model output back to the agent with an explicit correction instead of raising `OutputParserException`.

In [ ]:
tools = [search_tool]
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=(
        "Invalid ReAct format. Follow the format in the prompt exactly. "
        "If you already know the answer, respond with 'Final Answer: <answer>'."
    ),
    max_iterations=5,
)

In [ ]:
response = agent_executor.invoke({
    "input": "Tell me the latest news about Iran and the USA"
})

print(response["output"])